<a href="https://colab.research.google.com/github/braaaaayaaaaan/6001CMD_CW2_Group20/blob/main/PartC_Preprocessing_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part C: Data Preprocessing

Preparation of the LendingClub dataset for Logistic Regression, Random Forest and Histogram Gradient Boosting. The same processed datasets are used for all three models.


## 1. Load the data

Load `loan.csv` from the `ML_Group` folder in Google Drive.


In [ ]:
# Method: pandas read_csv (load only required columns).
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

try:
    from google.colab import drive
    drive.mount('/content/drive')
    folder = Path('/content/drive/MyDrive/ML_Group')
except ImportError:
    folder = Path('ML_Group')  # For running locally.

file = folder / 'loan.csv'
assert file.exists(), f'Cannot find {file}'

# The next date-check cell estimates the dataset's observation cutoff.

# These are the files that will be saved in ML_Group.
output_files = {name: folder / f'{name}_dataset.csv'
                for name in ['train', 'validation', 'test']}
print('Dataset:', file)

Mounted at /content/drive
Dataset: /content/drive/MyDrive/ML_Group/loan.csv


## 2. Select features

Use the features selected during Part B (EDA). Remove highly redundant features and features with excessive missing values.


In [ ]:
# Method: manual feature selection based on Part B EDA.
numeric = [
    'loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti',
    'inq_last_6mths', 'open_acc', 'total_acc', 'revol_util', 'revol_bal',
    'delinq_2yrs', 'pub_rec_bankruptcies', 'mths_since_last_delinq',
    'pub_rec', 'mort_acc', 'funded_amnt', 'funded_amnt_inv',
    'acc_open_past_24mths', 'bc_open_to_buy', 'mo_sin_rcnt_tl', 'num_il_tl',
    'pct_tl_nvr_dlq', 'total_bal_ex_mort', 'total_bc_limit',
    'open_il_12m', 'il_util', 'inq_fi'
]
categorical = [
    'term', 'purpose', 'emp_length', 'home_ownership',
    'verification_status', 'grade', 'sub_grade'
]
drop_features = ['funded_amnt', 'funded_amnt_inv', 'grade',
                 'open_il_12m', 'il_util', 'inq_fi']
keep_numeric = [c for c in numeric if c not in drop_features]
keep_categorical = [c for c in categorical if c not in drop_features]
needed = ['id', 'loan_status', 'issue_d', 'earliest_cr_line'] + numeric + categorical

# Read the selected columns, not all 145 original columns.
df = pd.read_csv(file, usecols=needed, low_memory=False)
print('Original rows:', len(df))
print('Removed predictors:', drop_features)

Original rows: 2260668
Removed predictors: ['funded_amnt', 'funded_amnt_inv', 'grade', 'open_il_12m', 'il_util', 'inq_fi']


### Check the latest recorded dates

Display the latest issue, payment and credit-pull dates. Estimate the observation cutoff using the latest recorded activity. This is an estimate, not a verified dataset update date.


In [ ]:
# Method: date inspection (latest recorded activity).
# Read only the date columns to avoid loading the whole dataset again.
date_columns = ['issue_d', 'last_pymnt_d', 'last_credit_pull_d']
dates = pd.read_csv(file, usecols=date_columns, dtype='string')

latest_dates = {}
for col in date_columns:
    parsed = pd.to_datetime(dates[col], format='%b-%Y', errors='coerce')
    latest_dates[col] = parsed.max()
    print(f'{col}: {latest_dates[col]}')

# Estimate the observation cutoff from the latest recorded loan activity.
# Use the end of that month because the dates only specify month and year.
activity_dates = [latest_dates[col] for col in ['last_pymnt_d', 'last_credit_pull_d']]
activity_dates = [date for date in activity_dates if pd.notna(date)]
if not activity_dates:
    raise ValueError('No valid payment or credit-pull dates to estimate the cutoff.')

SNAPSHOT_DATE = max(activity_dates) + pd.offsets.MonthEnd(0)
print('Estimated observation cutoff:', SNAPSHOT_DATE.date())
print('Note: This is inferred from activity dates, not a verified update date.')

# Make sure the estimate is not earlier than the latest issue month.
if pd.notna(latest_dates['issue_d']) and SNAPSHOT_DATE < latest_dates['issue_d']:
    raise ValueError('Estimated cutoff is earlier than the latest loan issue date.')

issue_d: 2018-12-01 00:00:00
last_pymnt_d: 2019-02-01 00:00:00
last_credit_pull_d: 2019-03-01 00:00:00
Estimated observation cutoff: 2019-03-31
Note: This is inferred from activity dates, not a verified update date.


## 3. Filter loan outcomes and maturity

Keep loans with confirmed `Fully Paid` or `Charged Off` outcomes. Apply a common 60-month maturity window to reduce bias from recently issued loans.


In [ ]:
# Method: rule-based filtering (known outcomes + 60-month maturity check).
df['issue_month'] = pd.to_datetime(df['issue_d'], format='%b-%Y', errors='coerce')
df['term_months'] = pd.to_numeric(
    df['term'].astype('string').str.extract(r'(\d+)', expand=False), errors='coerce'
)

# Check the status distribution before removing rows.
print(df['loan_status'].value_counts().head(12))
print('Latest issue month:', df['issue_month'].max())
snapshot = pd.Timestamp(SNAPSHOT_DATE)
assert snapshot >= df['issue_month'].max(), 'Snapshot date is earlier than the latest issue month.'

# Use the end of the issue month for a conservative maturity check.
mature = (df['issue_month'] + pd.offsets.MonthEnd(0) +
          pd.DateOffset(months=60) <= snapshot)
valid = (df['issue_month'].dt.year.between(2007, 2017) &
         df['term_months'].isin([36, 60]))
resolved = df['loan_status'].isin(['Fully Paid', 'Charged Off'])
print('Mature loans still unresolved:', int((valid & mature & ~resolved).sum()))
df = df.loc[valid & mature & resolved].copy()
df['target'] = df['loan_status'].eq('Charged Off').astype('int8')
print('Eligible loans:', len(df))
print(df['target'].value_counts())

loan_status
Fully Paid                                             1041952
Current                                                 919695
Charged Off                                             261655
Late (31-120 days)                                       21897
In Grace Period                                           8952
Late (16-30 days)                                         3737
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     31
Name: count, dtype: int64
Latest issue month: 2018-12-01 00:00:00
Mature loans still unresolved: 3948
Eligible loans: 274178
target
0    231100
1     43078
Name: count, dtype: int64


## 4. Check for duplicates

Count identical rows, repeated loan IDs and missing IDs. Display examples before removing duplicates.


In [ ]:
# Method: duplicate detection (check first, do not remove yet).
df['id'] = df['id'].astype('string').str.strip().replace('', pd.NA)

# Count fully identical rows and repeated loan IDs.
print('Total rows:', len(df))
print('Fully identical rows:', int(df.duplicated().sum()))
print('Repeated loan IDs:', int(df.loc[df['id'].notna(), 'id'].duplicated().sum()))
print('Missing loan IDs:', int(df['id'].isna().sum()))

# Display a few records with repeated IDs for inspection.
repeated = df.loc[df['id'].notna() & df['id'].duplicated(keep=False)]
print('Rows belonging to repeated IDs:', len(repeated))
if not repeated.empty:
    display(repeated[['id', 'loan_status', 'issue_d', 'loan_amnt']].head(10))

# Check if any repeated IDs have different information.
conflicting_ids = []
for loan_id, group in repeated.groupby('id'):
    if len(group.drop_duplicates()) > 1:
        conflicting_ids.append(loan_id)
print('IDs with conflicting records:', len(conflicting_ids))
if conflicting_ids:
    print('Example conflicting IDs:', conflicting_ids[:10])


Total rows: 274178
Fully identical rows: 0
Repeated loan IDs: 0
Missing loan IDs: 274178
Rows belonging to repeated IDs: 0
IDs with conflicting records: 0


### Remove duplicates and check invalid records

Remove repeated records with matching loan IDs. Stop if records sharing an ID contain conflicting information. Check essential loan values.


In [ ]:
# Method: ID-based duplicate removal.
# Stop if the same loan ID has different records; inspect those first.
if conflicting_ids:
    raise ValueError('Conflicting loan IDs found. Check the examples above before continuing.')

before = len(df)
df = df.loc[~(df['id'].notna() & df['id'].duplicated())].copy()
print('Duplicate loan IDs removed:', before - len(df))

# Method: rule-based cleaning of essential loan information.
for col in ['loan_amnt', 'installment', 'int_rate']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
invalid = (df['loan_amnt'].le(0) | df['installment'].le(0) |
           df['int_rate'].lt(0) | df[['loan_amnt', 'installment', 'int_rate']].isna().any(axis=1))
print('Invalid contract records removed:', int(invalid.sum()))
df = df.loc[~invalid].copy()
assert len(df) >= 5000 and df['target'].nunique() == 2


Duplicate loan IDs removed: 0
Invalid contract records removed: 0


## 5. Split the dataset

Use a chronological split based on issue month. Earlier loans form the training set; later loans form the validation and test sets.


In [ ]:
# Method: chronological holdout split (approximately 70/15/15, by issue month).
counts = df.groupby('issue_month').size().sort_index()
months = counts.index
cumulative = counts.cumsum().to_numpy()
train_i = np.searchsorted(cumulative, len(df) * 0.70)
val_i = np.searchsorted(cumulative, len(df) * 0.85)
assert 1 <= train_i < val_i < len(months) - 1, 'Not enough distinct months for the split.'
train_end, val_end = months[train_i], months[val_i]

parts = {
    'train': df.loc[df['issue_month'] <= train_end].copy(),
    'validation': df.loc[(df['issue_month'] > train_end) &
                         (df['issue_month'] <= val_end)].copy(),
    'test': df.loc[df['issue_month'] > val_end].copy(),
}
for name, part in parts.items():
    assert part['target'].nunique() == 2, f'{name} contains only one class.'
    print(name, len(part), 'Charge-off rate:', round(part['target'].mean()*100, 2), '%')
del df  # Free memory before creating the processed features.

train 198221 Charge-off rate: 15.52 %
validation 45302 Charge-off rate: 15.53 %
test 30655 Charge-off rate: 17.18 %


## 6. Clean numerical values

Treat invalid negative values as missing. Retain financially plausible extreme values.


In [ ]:
# Method: rule-based cleaning; invalid negatives become missing (NaN).
nonnegative = [
    'annual_inc', 'dti', 'revol_util', 'revol_bal', 'inq_last_6mths',
    'open_acc', 'total_acc', 'delinq_2yrs', 'pub_rec_bankruptcies',
    'mths_since_last_delinq', 'pub_rec', 'mort_acc',
    'acc_open_past_24mths', 'mo_sin_rcnt_tl', 'num_il_tl',
    'total_bal_ex_mort', 'total_bc_limit'
]

for name, part in parts.items():
    for col in keep_numeric:
        part[col] = pd.to_numeric(part[col], errors='coerce')
    part[nonnegative] = part[nonnegative].mask(part[nonnegative] < 0)
    part['pct_tl_nvr_dlq'] = part['pct_tl_nvr_dlq'].where(
        part['pct_tl_nvr_dlq'].between(0, 100))
    # Revolving utilisation above 100% may be real, so do not cap it.
    print(name, 'Missing DTI:', int(part['dti'].isna().sum()))

train Missing DTI: 0
validation Missing DTI: 0
test Missing DTI: 0


## 7. Create new features

Create affordability ratios, credit-history measures and delinquency indicators from the selected features.


In [ ]:
# Method: feature engineering (ratios, indicators, ordinal encoding, log1p).
def add_features(part):
    x = part.copy()
    earliest = pd.to_datetime(x['earliest_cr_line'], format='%b-%Y', errors='coerce')
    months = (x['issue_month'].dt.year - earliest.dt.year) * 12 + (x['issue_month'].dt.month - earliest.dt.month)
    x['credit_history_months'] = months.where(months >= 0)

    # Avoid division by zero for income-based ratios.
    income = x['annual_inc'].where(x['annual_inc'] > 0)
    x['payment_to_income'] = 12 * x['installment'] / income
    x['loan_to_income'] = x['loan_amnt'] / income
    x['nonmortgage_balance_to_income'] = x['total_bal_ex_mort'] / income
    x['income_zero_or_missing'] = income.isna().astype(int)

    x['recent_delinquency_flag'] = x['delinq_2yrs'].fillna(0).gt(0).astype(int)
    x['delinq_2yrs_missing'] = x['delinq_2yrs'].isna().astype(int)
    x['delinq_interval_recorded'] = x['mths_since_last_delinq'].notna().astype(int)
    x['term_60m'] = x['term_months'].eq(60).astype(int)

    # Convert A1 to G5 into an ordered number from 1 to 35.
    grade = x['sub_grade'].astype('string').str.strip().str.upper().str.extract(r'^([A-G])([1-5])$')
    x['subgrade_rank'] = grade[0].map(dict(zip('ABCDEFG', range(7)))) * 5 + pd.to_numeric(grade[1])
    mismatch = (x['grade'].notna() & grade[0].notna() &
                x['grade'].astype('string').str.strip().str.upper().ne(grade[0]))
    x.loc[mismatch.fillna(False), 'subgrade_rank'] = np.nan

    # Log-transform skewed monetary variables without removing outliers.
    for col in ['annual_inc', 'revol_bal', 'total_bal_ex_mort', 'total_bc_limit']:
        x[col + '_log'] = np.log1p(x[col].where(x[col] >= 0))
    return x

parts = {name: add_features(part) for name, part in parts.items()}
print('Feature engineering complete.')

Feature engineering complete.


## 8. Handle missing values and encode features

Apply median imputation, one-hot encoding and standardisation. Fit the transformations on training data only and apply them unchanged to validation and test data.


In [ ]:
# Methods: median imputation, StandardScaler, one-hot encoding.
# Fit on training data only; apply the same fitted steps to validation/test.
continuous = [
    'loan_amnt', 'int_rate', 'installment', 'dti', 'inq_last_6mths',
    'open_acc', 'total_acc', 'revol_util', 'delinq_2yrs',
    'pub_rec_bankruptcies', 'mths_since_last_delinq', 'pub_rec', 'mort_acc',
    'acc_open_past_24mths', 'bc_open_to_buy', 'mo_sin_rcnt_tl',
    'num_il_tl', 'pct_tl_nvr_dlq', 'annual_inc_log', 'revol_bal_log',
    'total_bal_ex_mort_log', 'total_bc_limit_log', 'credit_history_months',
    'payment_to_income', 'loan_to_income', 'nonmortgage_balance_to_income',
    'subgrade_rank'
]
binary = ['term_60m', 'recent_delinquency_flag', 'delinq_2yrs_missing',
          'delinq_interval_recorded', 'income_zero_or_missing']
nominal = ['purpose', 'emp_length', 'home_ownership', 'verification_status']
features = continuous + binary + nominal

# Keep missing categories as Unknown and avoid infinite values.
for part in parts.values():
    part[continuous] = part[continuous].replace([np.inf, -np.inf], np.nan)
    for col in nominal:
        part[col] = part[col].astype('string').str.strip().replace('', pd.NA).fillna('Unknown')

# Check for columns that have no observed training values.
empty = [col for col in continuous if parts['train'][col].notna().sum() == 0]
assert not empty, f'Training data has no values for: {empty}'

preprocessor = ColumnTransformer([
    ('numbers', Pipeline([('fill', SimpleImputer(strategy='median')),
                          ('scale', StandardScaler())]), continuous),
    ('binary', 'passthrough', binary),
    ('categories', OneHotEncoder(handle_unknown='ignore', sparse_output=False,
                                 dtype=np.float32), nominal)
], verbose_feature_names_out=False)
preprocessor.fit(parts['train'][features])
columns = preprocessor.get_feature_names_out().tolist()
print('Processed feature count:', len(columns))

Processed feature count: 66


## 9. Save the datasets

Save separate training, validation and test CSV files in `ML_Group`, using identical feature columns.


In [ ]:
# Method: batch CSV export (same columns for every model).
for name, part in parts.items():
    path = output_files[name]
    first = True
    for start in range(0, len(part), 50000):
        batch = part.iloc[start:start + 50000]
        values = preprocessor.transform(batch[features])
        assert np.isfinite(values).all(), f'{name} has missing or infinite values.'
        result = pd.DataFrame(values, columns=columns)
        result['target'] = batch['target'].to_numpy()
        result.to_csv(path, mode='w' if first else 'a',
                      header=first, index=False, float_format='%.9g')
        first = False
    print('Saved:', path, '| Rows:', len(part))

# Save the fitted preprocessing steps for later use if needed.
joblib.dump(preprocessor, folder / 'shared_preprocessor.joblib')

Saved: /content/drive/MyDrive/ML_Group/train_dataset.csv | Rows: 198221
Saved: /content/drive/MyDrive/ML_Group/validation_dataset.csv | Rows: 45302
Saved: /content/drive/MyDrive/ML_Group/test_dataset.csv | Rows: 30655


['/content/drive/MyDrive/ML_Group/shared_preprocessor.joblib']

## 10. Final checks and class weights

Check dataset columns and calculate balanced class weights from training labels. No SMOTE or resampling is applied.


In [ ]:
# Method: balanced class weights (no SMOTE or other resampling).
headers = [pd.read_csv(path, nrows=0).columns.tolist() for path in output_files.values()]
assert headers[0] == headers[1] == headers[2]
assert headers[0][-1] == 'target'
assert 'id' not in headers[0] and 'loan_status' not in headers[0]

# Calculate class weights using the training labels only.
y = parts['train']['target'].to_numpy()
weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=y)
class_weights = dict(zip([0, 1], weights.tolist()))
print('Training class weights:', class_weights)
print('All three datasets have the same features.')
print('Target: 0 = Fully Paid, 1 = Charged Off')

Training class weights: {0: 0.591888227987196, 1: 3.220696714652455}
All three datasets have the same features.
Target: 0 = Fully Paid, 1 = Charged Off


## Summary of preprocessing methods

| Step | Method |
|---|---|
| Feature selection | EDA-based manual selection and removal of redundant/high-missingness features |
| Loan selection | Rule-based filtering of known outcomes and mature loans |
| Duplicates | Duplicate detection using loan `id` |
| Train/validation/test | Chronological holdout split, approximately 70% / 15% / 15% |
| Invalid values | Rule-based cleaning; invalid values set to missing |
| Missing numerical values | **Median imputation** (`SimpleImputer`) fitted on training data |
| Missing categories | **Unknown** category |
| Skewed monetary features | **Log transformation** (`log1p`) |
| New features | Credit history, affordability ratios, and binary indicators |
| Ordered risk grades | **Ordinal encoding** (`subgrade_rank`) |
| Other categorical features | **One-hot encoding** (`OneHotEncoder`) |
| Numerical scaling | **Z-score standardisation** (`StandardScaler`) |
| Class imbalance | **Balanced class weights**, calculated from training labels; **no SMOTE/resampling** |
| Dimensionality reduction | **None** (no PCA) |
| Output | Three CSV files with identical processed features |

All learned preprocessing values come from the training split only. The model training code must actually **apply** the class weights; this notebook only calculates them. Threshold selection and F2 evaluation happen in the modelling notebooks.